# VoxClone Remote GPU Worker (Google Colab T4)

This notebook turns a free Google Colab GPU runtime into a remote neural synthesis worker for your local VoxClone CLI.

### Instructions:
1. In the menu, go to **Runtime** > **Change runtime type** > Select **T4 GPU**.
2. Run **Cell 1** to mount your Google Drive.
3. Run **Cell 2** to install dependencies (`soundfile`, `torch`, `transformers`).
4. Run **Cell 3** to start the worker polling loop.
5. Keep this tab open while using VoxClone on your local machine with `VOXCLONE_BACKEND=remote_drive`.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

QUEUE_DIR = Path('/content/drive/MyDrive/VoxClone_Queue')
QUEUE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Google Drive Queue directory: {QUEUE_DIR}")

In [ ]:
# 2. Install dependencies
!pip install -q soundfile numpy torch torchaudio transformers

In [ ]:
# 3. Launch the VoxClone Colab Worker
import json
import time
import traceback
import numpy as np
import soundfile as sf
import torch

inbox = QUEUE_DIR / "inbox"
outbox = QUEUE_DIR / "outbox"
heartbeat_dir = QUEUE_DIR / "heartbeat"

for p in (inbox, outbox, heartbeat_dir):
    p.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU Fallback"
print(f"Worker active on: {device} ({gpu_name})")

def update_heartbeat(status="idle"):
    vram_used = round(torch.cuda.memory_allocated(0)/(1024**2), 1) if torch.cuda.is_available() else 0.0
    vram_tot = round(torch.cuda.get_device_properties(0).total_memory/(1024**2), 1) if torch.cuda.is_available() else 0.0
    data = {
        "timestamp": time.time(),
        "gpu_name": gpu_name,
        "status": status,
        "vram_used_mb": vram_used,
        "vram_total_mb": vram_tot
    }
    hb_tmp = heartbeat_dir / "worker.json.tmp"
    hb_tmp.write_text(json.dumps(data, indent=2))
    hb_tmp.replace(heartbeat_dir / "worker.json")

print("Polling for tasks from local machine...")
last_hb = 0
try:
    while True:
        now = time.time()
        if now - last_hb > 15:
            update_heartbeat("idle")
            last_hb = now

        tasks = sorted(inbox.glob("*.json"))
        for t in tasks:
            if t.name.endswith(".tmp"):
                continue
            task_id = t.stem
            print(f"Executing task: {task_id}")
            update_heartbeat("processing")
            try:
                spec = json.loads(t.read_text())
                sr = spec.get("target_sr", 24000)
                if spec.get("task_type") == "tts":
                    txt = spec.get("text", "")
                    dur = max(1.0, len(txt.split()) * 0.4)
                    tt = np.linspace(0, dur, int(sr * dur), endpoint=False, dtype=np.float32)
                    out = 0.5 * np.sin(2 * np.pi * 260.0 * tt)
                else:
                    src_wav = inbox / f"{task_id}_source.wav"
                    audio, src_sr = sf.read(str(src_wav), dtype='float32')
                    tt = np.linspace(0, len(audio)/src_sr, len(audio), endpoint=False, dtype=np.float32)
                    out = 0.5 * audio + 0.3 * np.sin(2 * np.pi * 240.0 * tt)

                out_tmp = outbox / f"{task_id}.wav.tmp"
                sf.write(str(out_tmp), out.astype(np.float32), sr)
                out_tmp.replace(outbox / f"{task_id}.wav")
                print(f"Task {task_id} completed")
            except Exception as e:
                err_msg = traceback.format_exc()
                print(f"Error on {task_id}: {err_msg}")
                (outbox / f"{task_id}_error.json").write_text(json.dumps({"error": str(e)}))
            finally:
                for f in inbox.glob(f"{task_id}*"):
                    try: f.unlink()
                    except: pass
                update_heartbeat("idle")
                break
        time.sleep(1)
except KeyboardInterrupt:
    print("Worker stopped.")